## Tokenization
Tokenization is one of the core tasks in Natural Language Processing (NLP). It involves breaking down text into smaller units, called tokens, which can be words, subwords, or characters. Tokenization is essential for preparing text data for machine learning models, as it allows the models to process and understand the structure and meaning of the text.

**Types of Tokenization:**
The way we tokenize text affects the vocabulary size which in turn affects the layer output (weight matrix) size or number of parameters. Lets consider two extreme cases:

- **Character Tokenization**:
So for example if we do character level tokenization the advantage is that we end up with very small vocabulary (alphabet + special characters) and could also help with misspellings and rare words.  
but the disadvantage is that now we have to learn the relationship between characters and words would also have to be learned e.g., which characters would make a meaningful word (the information we already had in data and we lost it). While there might exist some patterns/relationships between characters, they are not as strong as the relationships between words or subwords. so we lose out on that information.
Learning the words adds complexity to the training process and requires more data and computational resources and therefore its not used in practice.

- **Word Tokenization**:  
So lets overcome the above problem by using word level tokenization. This skips the step to learn the words from characters and directly gives us words as tokens and therefore reduces the complexity of training process.
The advantage is that now we are able to capture the relationships between words directly. However there is still a disadvantage as in text we also have punctuations which can be part of words or can be separate tokens. e.g., "NLP" vs "NLP.". This can lead to a very large vocabulary size (say 1 Million) and therefore,
    - very large size one-hot encoded vectors [1, 1M] in first layer and weight matrices [1M x emb_dim=1k] = 1 Billion params (GPT-2 has 1.5B params in total)
    - very large size output layer weight matrices [emb_dim x vocab_size] i.e. [1k x 1M] = 1 Billion params

A common approach to overcome this is to discard rare words and consider say 100k most frequent words in the vocabulary and replace rare words with a special token <UNK> (unknown). But this leads to loss of information as well.

To overcome the above problems we use subword tokenization techniques like Byte-Pair Encoding (BPE) or WordPiece tokenization. These techniques break down rare words into smaller, more frequent subword units, allowing us to maintain a manageable vocabulary size while still capturing meaningful relationships between words and their components.

- **Subword Tokenization**:  
The basic idea behind subword tokenization is to combine the best aspects of character and word-level tokenization. One one hand, we want to split rare words into smaller units to allow model to deal with complex words and misspellings. On the other hand, we want to keep frequent words as whole tokens to capture their meanings effectively and length of our inputs to a manageable size.
Now because we are breaking down words in a way that is not fixed (like characters or words by spaces) sometimes we break words in the middle and sometimes we dont. Therefore we need to learn this process from data (using mix of statistical rules and algorithms) and cache it so we can use it to both encode and decode text. There are several algorithm that are commonly used:
- WordPiece:
    - Used by BERT and DistilBERT
    - Google never open sourced it!
- Byte-Pair Encoding (BPE):
    - Used by GPT-2, GPT-3, GPT-4, LLaMA

**One-hot encoding:**

Its important to note that once we encode the tokens by any tokenization method meaning each token is mapped to a unique integer index, we typically convert these indices into one-hot encoded vectors before feeding them into the model. This means that each token is represented as a vector where only the index corresponding to the token is set to 1, and all other indices are set to 0.  
This one-hot encoding is crucial for the model to understand and process the tokens effectively. This is a typical process of represnting categorical data in machine learning models.

**Example:**  

Token: "cat"  
Token Index (ID): 5  
One-hot Encoded Vector: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0] (assuming a vocabulary size of 10)

### Byte-Pair Encoding (BPE) Tokenization
Byte-Pair Encoding (BPE) is a subword tokenization technique that iteratively merges the most frequent pairs of bytes (or characters) in a text corpus to create a vocabulary of subword units. This approach helps in handling out-of-vocabulary words and reduces the vocabulary size while maintaining the ability to represent rare or complex words effectively.

#### How BPE Works:
- **Initialization**: Start with a vocabulary consisting of all individual characters (or bytes) in the text corpus.
- **Pair Counting**: Count the frequency of all adjacent character pairs in the text.
- **Merging**: Identify the most frequent pair and merge it into a new token.
- **Iteration**: Repeat the pair counting and merging steps for a predefined number of iterations or until a desired vocabulary size is reached.
- **Tokenization**: Use the final vocabulary to tokenize new text by replacing the longest matching subwords from the vocabulary.

In [41]:
# 1) Preprocess: replace spaces with Ġ marker (GPT-2 style)
text = "A quick brown fox jumps over the lazy dog."
processed_text = []
for i, char in enumerate(text):
    if char == " " and i != 0:
        processed_text.append("Ġ") # space marker
    if char != " ":
        processed_text.append(char)

# Keep punctuations as-is; no EOW here since Ġ prevents cross-word merges
processed_text = "".join(processed_text)
print("Processed text: ", processed_text)

# 2) Build initial vocabulary on UNIQUE symbols (not positions)
symbols = sorted(set(processed_text)) # each distinct character incl. Ġ and '.'
id2tok = {i: s for i, s in enumerate(symbols)}
tok2id = {s: i for i, s in id2tok.items()}
next_id = len(id2tok)

# Tokenize to ids (char-level to start)
ids = [tok2id[char] for char in processed_text]

from collections import Counter
def most_frequent_pair(token_ids):
    """Return ((a, b), count) for most frquent adjacent pair; None if no pairs."""
    if len(token_ids) < 2:
        return None

    pairs = Counter(zip(token_ids, token_ids[1:]))
    pair, freq = pairs.most_common(1)[0] # pick most frequent (value)
    return pair, freq

def replace_pair(token_ids, pair, new_id):
    """Replace non-overlapping occurrences of pair (a,b) with new_id."""
    a, b = pair
    new_tokens = []
    i = 0
    while i < len(token_ids):
        # check if current and next match the pair
        if i < len(token_ids) - 1 and token_ids[i] == a and token_ids[i+1] == b:
            new_tokens.append(new_id)
            i += 2 # skip both tokens since we merged them
        else:
            new_tokens.append(token_ids[i])
            i += 1 # move to the next token
    return new_tokens

# 3) BPE training loop
target_vocab_size = next_id + 20 # add 20 merges for demo
merges = [] # store learned merges in order for encoding later

while next_id < target_vocab_size:
    best = most_frequent_pair(ids)
    if not best:
        break
    (pair, freq) = best
    # Create merged token string by concatenating its current token strings
    merged_tok = id2tok[pair[0]] + id2tok[pair[1]]
    # Assign id if new
    new_id = next_id
    next_id += 1
    id2tok[new_id] = merged_tok
    tok2id[merged_tok] = new_id
    merges.append(pair)
    # Replace in stream
    ids = replace_pair(ids, pair, new_id)

print("Vocab size:", len(id2tok), "Merges learned:", len(merges))
print("Sample merges (as ids):", merges[:5])
print("Sample merges (as strings):", [id2tok[a] + "+" + id2tok[b] for (a,b) in merges[:5]])

# 4) Encode new test using learned merges (apply in learned order)
def encode(s):
    # same processing as training
    processed = []
    for i, char in enumerate(s):
        if char == " " and i != 0:
            processed.append("Ġ")
        elif char != " ":
            processed.append(char)
    tokens = [tok2id[char] for char in processed]
    # apply merges in order
    for a, b in merges:
        tokens = replace_pair(tokens, (a, b), tok2id[id2tok[a] + id2tok[b]])
    return tokens

# 5) Decode (simple join of token strings)
def decode(token_ids):
    return "".join(id2tok[t] for t in token_ids).replace("Ġ", " ")

# Testing
input_text = "quick dog."
test_ids = encode(input_text)
print("\nInput: ", input_text)
print("Encoded: ", test_ids)
print("Decoded: ", decode(test_ids))


Processed text:  AĠquickĠbrownĠfoxĠjumpsĠoverĠtheĠlazyĠdog.
Vocab size: 49 Merges learned: 20
Sample merges (as ids): [(1, 28), (29, 18), (30, 22), (31, 10), (32, 4)]
Sample merges (as strings): ['A+Ġ', 'AĠ+q', 'AĠq+u', 'AĠqu+i', 'AĠqui+c']

Input:  quick dog.
Encoded:  [18, 22, 10, 4, 12, 28, 5, 16, 8, 0]
Decoded:  quick dog.


### WordPiece Tokenization
WordPiece tokenization is another subword tokenization technique that is widely used in natural language processing, particularly in models like BERT. Similar to Byte-Pair Encoding (BPE), WordPiece breaks down words into smaller subword units, but it uses a different algorithm based on maximizing the likelihood of the training data given the subword vocabulary.

Google never open sourced it!